In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

# 1. Load and Preprocess the dataset
# FIX: Added sep='\t' and names=['label', 'text'] because the file is tab-separated and has no header
df = pd.read_csv("spamhamdata.csv", sep='\t', names=['label', 'text'])

# Check for null values
print("--- Null Value Check ---")
print(df.isnull().sum())

# Check for duplicates
# 1. Check duplicates initially
initial_dupes = df.duplicated(subset=['text']).sum()
print(f"Duplicates before cleaning: {initial_dupes}")

# 2. Remove duplicates
df = df.drop_duplicates(subset=['text'], keep='first')

# 3. VERIFY: Check duplicates again (Should be 0)
final_dupes = df.duplicated(subset=['text']).sum()
print(f"Duplicates after cleaning: {final_dupes}")
print(f"Total unique emails remaining: {len(df)}")

# FIX: Map string labels 'ham'/'spam' to numbers 0 and 1
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

X_texts = df['text']
y = df['label_num']

# 2. Split with Stratification
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_texts, 
    y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

# 3. Vectorize (Bag of Words)
vectorizer = CountVectorizer(stop_words='english', max_features=1000)
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

# 4. Train the Naive Bayes Classifier
model = MultinomialNB(alpha=0.1)
model.fit(X_train, y_train)

# 5. Evaluate the results
y_pred = model.predict(X_test)
print(f"\nModel Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# 6. Test with your specific 'fake' Retail Phishing emails
fake_emails = [
    "URGENT: Your retail invoice for $500 is overdue. Pay now.",
    "Congratulations! You've won a $1000 Walmart gift card! Click here.",
    "Verify your Amazon account password immediately to avoid suspension.",
    "Please find the attached report regarding the energy meeting tomorrow.",
    "Hey, are we still playing badminton this weekend at the college court?"
]

X_fake = vectorizer.transform(fake_emails)
predictions = model.predict(X_fake) 
probs = model.predict_proba(X_fake) # for probability instead of a direct 'spam' 'ham'

print("\n--- Manual Retail Test Results with Confidence ---")
for i, email in enumerate(fake_emails):
    pred = predictions[i]
    confidence = probs[i][pred] * 100 #converts probability into percentage
    label = "🚨 SPAM" if pred == 1 else "✅ HAM"
    print(f"[{label}] ({confidence:.2f}% confidence) -> {email}")

--- Null Value Check ---
label    0
text     0
dtype: int64
Duplicates before cleaning: 403
Duplicates after cleaning: 0
Total unique emails remaining: 5169

Model Accuracy: 98.16%

Detailed Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       903
        Spam       0.94      0.92      0.93       131

    accuracy                           0.98      1034
   macro avg       0.96      0.95      0.96      1034
weighted avg       0.98      0.98      0.98      1034


--- Manual Retail Test Results with Confidence ---
[🚨 SPAM] (98.14% confidence) -> URGENT: Your retail invoice for $500 is overdue. Pay now.
[🚨 SPAM] (100.00% confidence) -> Congratulations! You've won a $1000 Walmart gift card! Click here.
[🚨 SPAM] (77.82% confidence) -> Verify your Amazon account password immediately to avoid suspension.
[✅ HAM] (99.93% confidence) -> Please find the attached report regarding the energy meeting tomorrow.
[✅ HAM] (99.94

In [8]:
df

,label,text,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0
...,...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...,1
5568,ham,Will ü b going to esplanade fr home?,0
5569,ham,"Pity, * was in mood for that. So...any other s...",0
5570,ham,The guy did some bitching but I acted like i'd...,0
